# Shanghai T2DM chronological CGM forecasting

This notebook calls the same reusable PyTorch GRU pipeline as `train_t2dm.py`. It treats each workbook as its own patient record, preserves timestamp order, fits on the first 80% of timestamped rows, and tests only on the remaining final 20% of rows. Sliding windows use the preceding 24 readings to predict the next 15-minute CGM value.

The supplied workbooks vary in duration. Review `timeline_splits.csv` for each patient's actual training, validation, and test coverage. Test evaluation is rolling one-step-ahead, not a recursive fixed-day forecast.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

repository_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "models/training/t2dm/t2dm_pipeline.py").is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError("Open this notebook from inside the project repository.")
sys.path.insert(0, str(repository_root))

from models.training.t2dm.t2dm_pipeline import (
    DEFAULT_DATA_DIR, DEFAULT_OUTPUT_DIR, train_and_evaluate,
)

data_dir = DEFAULT_DATA_DIR
output_dir = DEFAULT_OUTPUT_DIR
print(f"Training data: {data_dir}")
print(f"Evaluation output: {output_dir}")


In [ ]:
result = train_and_evaluate(
    data_dir=data_dir,
    output_dir=output_dir,
    epochs=30,
    batch_size=128,
    sequence_length=24,
    hidden_dim=64,
    training_stride=4,
    seed=2026,
)
print(json.dumps(result["test_metrics"], indent=2))

In [ ]:
split_report = pd.read_csv(output_dir / "timeline_splits.csv")
split_report

In [ ]:
predictions = pd.read_csv(output_dir / "test_predictions.csv", parse_dates=["timestamp"])
patient_id = predictions["patient_id"].iloc[0]
sample = predictions.loc[predictions["patient_id"] == patient_id].head(96)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(sample["timestamp"], sample["actual_glucose_mg_dl"], label="Actual")
ax.plot(sample["timestamp"], sample["predicted_glucose_mg_dl"], label="GRU forecast")
ax.plot(sample["timestamp"], sample["persistence_baseline_mg_dl"], label="Persistence baseline", alpha=0.7)
ax.set(title=f"Held-out CGM prediction - patient {patient_id}", ylabel="Glucose (mg/dL)")
ax.legend()
fig.autofmt_xdate()
plt.show()

This is an educational research model, not a diagnosis or treatment/dosing tool. See `../README.md` and `../../../docs/training-model-migration.md` for training and deployment instructions.